In [6]:
from langchain.llms import OpenAIChat
from langchain import PromptTemplate, LLMChain
from langchain.callbacks import get_openai_callback

In [7]:
import os
from os.path import abspath, join
import dotenv

In [8]:
dir = abspath('')

In [9]:
dotenv.load_dotenv('../bin/.env')

True

In [44]:
with open(join(dir, 'test_data', '2303.02248.txt'), 'r') as f:
    full_text = f.read()

# categorize

In [72]:
options = ['chemicals', 'biochemical reactions', 'species',
           'countries', 'geological ages', 'ignition fuel',
           'planetary objects', 'NASA projects']

In [73]:
template = """The following text was extracted from a PDF document:

{text}

We are trying to categorize the document according to categories it
will discuss. The category options are:

{options}

Based on the text extract, list the categories that are likely to
exist in the full text of the article. List one category on each line.
Only provide categories from the category options list.
If no categories are expected, say 'None found'.
"""

In [74]:
prompt = PromptTemplate(template=template, input_variables=["text", "options"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
with get_openai_callback() as cb:
    # TODO how to get the right first text blob? should use embeddings
    print(llm_chain.run(text=full_text[2000:8000], options=', '.join(options)))
    print(f'used {cb.total_tokens} tokens, cost ${cb.total_tokens/1000 * 0.002:.3f}')



planetary objects
NASA projects
used 1482 tokens, cost $0.003


# find stuff

In [11]:
llm = OpenAIChat(temperature=0)

In [8]:
template = """The following text was extracted from a PDF document:

{text}

List the chemical compounds that are mentioned in the article. If you
find a chemical, provide the name of each chemical on a new line with
a description, like (chemical Name: Description). If you do not find a
chemical, say "No chemicals found".
"""
prompt = PromptTemplate(template=template, input_variables=["text"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for i in range(2):
    with get_openai_callback() as cb:
        print(llm_chain.run(full_text[2000*(i):2000*(i+1)]).replace("No chemicals found.", "").strip())
        print(cb)
        print(f'used {cb.total_tokens} tokens, cost ${cb.total_tokens/1000 * 0.002:.3f}')

vinblastine: an anti-cancer therapeutic
vincristine: an anti-cancer therapeutic
Monoterpene indole alkaloids (MIAs): a diverse family of complex plant secondary metabolites
vindoline: a precursor to vinblastine and catharanthine
catharanthine: a precursor to vinblastine and vindoline
geranyl pyrophosphate: a yeast native metabolite
tryptophan: a yeast native metabolite
used 738 tokens, cost $0.001

used 514 tokens, cost $0.001


In [42]:
template = """The following text was extracted from a PDF document:

{text}

List the organisms that are being studied in the article. If you
find an organism, provide the common name and scientific name of
each organism on a new line with, like:

- Common Name (Scientific Name).

If you do not find an organism, say "No organisms found".
"""
prompt = PromptTemplate(template=template, input_variables=["text"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for t in map(''.join, zip(*[iter(full_text)]*2000)):
    print(llm_chain.run(t).replace("No organisms found.", "").replace('-', '').strip())

Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)
 Plants (Gentianales plants, Catharanthus roseus)
Baker's yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)

Madagascar periwinkle (Catharanthus roseus)
 Yeast (Saccharomyces cerevisiae)



Yeast (Saccharomyces cerevisiae)
 Rauvolfia tetraphylla
 Sesamum indicum
 Catharanthus roseus
 Populus trichocarpa
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
 Indian snakeroot (Rauvolfia serpentina)
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)

Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)


Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Yeast (Saccharomyces cerevisiae)
 Madagascar periwinkle (Catharanthus roseus)
Mexican Tab

In [22]:
text = f"""The following text was extracted from a PDF document:

{full_text[1000:2000]}

List the model organisms being studies that are mentioned in the article. If you
find organisms, provide each organism scientific name on a new line with no other
information. If you do not find a model organism, say "No organisms found".
"""
print(llm(text))


Saccharomyces cerevisiae
Catharanthus roseus


In [30]:
full_text.index('SpyCas9')

29081

In [37]:
text = f"""The following text was extracted from a PDF document:

{full_text[29000:31000]}

List the engineered strains that are mentioned in the article. If you
find an engineered strain, provide the name of each strain on a new line with
a description, like (Strain Name: Description). If you do not find a
strain, say "No strains found".
"""
print(llm(text))


MIA-CM-3: De novo strictosidine production strain
MIA-CR-A: Strain producing 11.5 μg l−1 tabersonine
MIA-CW-1: Strain producing 0.137 μg l−1 vindoline
MIA-EM-1: Biphasic fermentation strain producing 2.32 μg l−1 vindoline and 2.82 μg l−1 catharanthine


In [15]:
paper_path = join(dir, '..', 'data', 's41586-022-05157-3.pdf')